# Customer Master Data Quality & Entity Matching Pipeline

This notebook generates synthetic B2B customer data from ERP, CRM, and e-commerce systems for data quality, deduplication, and entity matching analysis.



In [3]:
from pathlib import Path
import random
import re

import pandas as pd
from faker import Faker
from IPython.display import display


# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42
CUSTOMER_COUNT = 500

random_generator = random.Random(SEED)

fake = Faker("en_US")
Faker.seed(SEED)


def find_project_root():
    """Find the project folder from the notebook location."""
    current_path = Path.cwd().resolve()

    for candidate in [current_path, *current_path.parents]:
        if (
            (candidate / "data").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Project root not found. Confirm that the data and notebooks "
        "folders exist."
    )


PROJECT_ROOT = find_project_root()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# DATA VARIATION FUNCTIONS
# ============================================================

def maybe_blank(value, probability):
    """Return a blank value according to the given probability."""
    if random_generator.random() < probability:
        return ""

    return value


def remove_company_suffix(company_name):
    """Remove common legal suffixes from company names."""
    suffix_pattern = (
        r",?\s+"
        r"(LLC|INC\.?|INCORPORATED|CORPORATION|CORP\.?|"
        r"LTD\.?|LIMITED|PLC)$"
    )

    return re.sub(
        suffix_pattern,
        "",
        company_name,
        flags=re.IGNORECASE,
    ).strip()


def introduce_typo(value):
    """Remove one letter to simulate a typing error."""
    valid_positions = [
        index
        for index, character in enumerate(value)
        if character.isalpha()
    ]

    if len(valid_positions) < 6:
        return value

    position = random_generator.choice(valid_positions[1:-1])

    return value[:position] + value[position + 1:]


def abbreviate_address(address):
    """Apply common United States address abbreviations."""
    replacements = {
        "Street": "St",
        "Avenue": "Ave",
        "Road": "Rd",
        "Boulevard": "Blvd",
        "Drive": "Dr",
        "Lane": "Ln",
        "Highway": "Hwy",
        "Parkway": "Pkwy",
    }

    for original, abbreviated in replacements.items():
        address = re.sub(
            rf"\b{original}\b",
            abbreviated,
            address,
            flags=re.IGNORECASE,
        )

    return address


def format_phone(phone, source_system):
    """Format the same phone number differently by source."""
    digits = re.sub(r"\D", "", phone)

    formats = {
        "ERP": f"{digits[:3]}-{digits[3:6]}-{digits[6:]}",
        "CRM": f"({digits[:3]}) {digits[3:6]}-{digits[6:]}",
        "ECOMMERCE": f"+1 {digits[:3]} {digits[3:6]} {digits[6:]}",
    }

    return formats[source_system]


def create_email(company_name, entity_number):
    """Create a reproducible synthetic company email."""
    domain = company_name.lower()

    domain = re.sub(
        r"\b(llc|inc|corporation|corp|ltd|company|co|plc)\b",
        "",
        domain,
    )

    domain = re.sub(r"[^a-z0-9]", "", domain)[:20]
    domain = domain or "company"

    return f"contact@{domain}{entity_number}.com"


# ============================================================
# BASE CUSTOMER CREATION
# ============================================================

def create_base_customer(entity_number):
    """Create a clean synthetic B2B customer."""
    company_name = fake.unique.company()

    phone = (
        f"{random_generator.randint(201, 989):03d}"
        f"{random_generator.randint(200, 999):03d}"
        f"{random_generator.randint(0, 9999):04d}"
    )

    tax_id = (
        f"{random_generator.randint(10, 99):02d}-"
        f"{random_generator.randint(0, 9_999_999):07d}"
    )

    return {
        "ground_truth_id": f"GT-{entity_number:06d}",
        "company_name": company_name,
        "tax_id": tax_id,
        "email": create_email(company_name, entity_number),
        "phone": phone,
        "address_line_1": fake.street_address(),
        "city": fake.city(),
        "state": fake.state_abbr(),
        "postal_code": fake.zipcode(),
    }


# ============================================================
# SOURCE-SPECIFIC RECORD CREATION
# ============================================================

def create_source_record(base_customer, source_system, entity_number):
    """Create a potentially inconsistent source record."""
    company_name = base_customer["company_name"]
    tax_id = base_customer["tax_id"]
    email = base_customer["email"]
    phone = format_phone(base_customer["phone"], source_system)
    address = base_customer["address_line_1"]
    city = base_customer["city"]
    state = base_customer["state"]
    postal_code = base_customer["postal_code"]

    if source_system == "CRM":
        if random_generator.random() < 0.65:
            company_name = remove_company_suffix(company_name)

        if random_generator.random() < 0.12:
            company_name = introduce_typo(company_name)

        if random_generator.random() < 0.30:
            company_name = company_name.upper()

        address = abbreviate_address(address)

        tax_id = maybe_blank(tax_id, 0.75)
        email = maybe_blank(email, 0.08)
        phone = maybe_blank(phone, 0.10)
        postal_code = maybe_blank(postal_code, 0.05)

        if tax_id and random_generator.random() < 0.40:
            tax_id = re.sub(r"\D", "", tax_id)

        if email and random_generator.random() < 0.20:
            email = email.upper()

    elif source_system == "ECOMMERCE":
        if random_generator.random() < 0.75:
            company_name = remove_company_suffix(company_name)

        if random_generator.random() < 0.20:
            company_name = introduce_typo(company_name)

        if random_generator.random() < 0.50:
            company_name = company_name.upper()

        address = abbreviate_address(address)

        if random_generator.random() < 0.25:
            address = address.upper()

        tax_id = maybe_blank(tax_id, 0.95)
        email = maybe_blank(email, 0.05)
        phone = maybe_blank(phone, 0.20)
        city = maybe_blank(city, 0.05)
        state = maybe_blank(state, 0.03)
        postal_code = maybe_blank(postal_code, 0.10)

        if tax_id and random_generator.random() < 0.60:
            tax_id = re.sub(r"\D", "", tax_id)

        if email and random_generator.random() < 0.30:
            email = email.upper()

    prefixes = {
        "ERP": "ERP",
        "CRM": "CRM",
        "ECOMMERCE": "ECM",
    }

    return {
        "ground_truth_id": base_customer["ground_truth_id"],
        "source_system": source_system,
        "source_customer_id": (
            f"{prefixes[source_system]}-{entity_number:06d}"
        ),
        "company_name": company_name,
        "tax_id": tax_id,
        "email": email,
        "phone": phone,
        "address_line_1": address,
        "city": city,
        "state": state,
        "postal_code": postal_code,
        "updated_at": fake.date_between(
            start_date="-2y",
            end_date="today",
        ).isoformat(),
    }


# ============================================================
# DATASET GENERATION
# ============================================================

datasets = {
    "ERP": [],
    "CRM": [],
    "ECOMMERCE": [],
}

for entity_number in range(1, CUSTOMER_COUNT + 1):
    base_customer = create_base_customer(entity_number)

    datasets["ERP"].append(
        create_source_record(
            base_customer,
            "ERP",
            entity_number,
        )
    )

    if random_generator.random() < 0.82:
        datasets["CRM"].append(
            create_source_record(
                base_customer,
                "CRM",
                entity_number,
            )
        )

    if random_generator.random() < 0.65:
        datasets["ECOMMERCE"].append(
            create_source_record(
                base_customer,
                "ECOMMERCE",
                entity_number,
            )
        )


# ============================================================
# CSV EXPORT
# ============================================================

output_files = {
    "ERP": "erp_customers.csv",
    "CRM": "crm_customers.csv",
    "ECOMMERCE": "ecommerce_customers.csv",
}

dataframes = {}

for source_system, filename in output_files.items():
    dataframe = pd.DataFrame(datasets[source_system])
    output_path = RAW_DATA_DIR / filename

    dataframe.to_csv(
        output_path,
        index=False,
        encoding="utf-8",
    )

    dataframes[source_system] = dataframe


# ============================================================
# GENERATION SUMMARY
# ============================================================

summary = pd.DataFrame(
    [
        {
            "source_system": source_system,
            "records": len(dataframe),
            "missing_tax_id": dataframe["tax_id"].eq("").sum(),
            "missing_email": dataframe["email"].eq("").sum(),
            "missing_phone": dataframe["phone"].eq("").sum(),
        }
        for source_system, dataframe in dataframes.items()
    ]
)

print("Datasets generated successfully.")
print(f"Output folder: {RAW_DATA_DIR}")

display(summary)


# ============================================================
# DUPLICATE EXAMPLES
# ============================================================

all_records = pd.concat(
    dataframes.values(),
    ignore_index=True,
)

sample_ids = (
    dataframes["ECOMMERCE"]["ground_truth_id"]
    .head(3)
    .tolist()
)

sample_records = all_records[
    all_records["ground_truth_id"].isin(sample_ids)
][
    [
        "ground_truth_id",
        "source_system",
        "company_name",
        "tax_id",
        "email",
        "phone",
        "address_line_1",
    ]
].sort_values(
    ["ground_truth_id", "source_system"]
)

display(sample_records)

Datasets generated successfully.
Output folder: C:\Users\jfraga\Desktop\py\Customer Master Data Quality & Entity Matching Pipeline\data\raw


,source_system,records,missing_tax_id,missing_email,missing_phone
0,ERP,500,0,0,0
1,CRM,406,313,31,42
2,ECOMMERCE,331,313,15,61


,ground_truth_id,source_system,company_name,tax_id,email,phone,address_line_1
500,GT-000001,CRM,"Rodriguez, Figueroa and Sanchez",,contact@rodriguezfigueroaand1.com,,181 Johnson Course
906,GT-000001,ECOMMERCE,"Rodriguez, Figueroa and Sanchez",,CONTACT@RODRIGUEZFIGUEROAAND1.COM,,181 JOHNSON COURSE
0,GT-000001,ERP,"Rodriguez, Figueroa and Sanchez",45-4108603,contact@rodriguezfigueroaand1.com,855-314-0409,181 Johnson Course
502,GT-000003,CRM,"MARTINEZ, NIELSEN AND MILLER",,contact@martineznielsenandmi3.com,(834) 570-9459,959 Janet Cape Apt. 413
907,GT-000003,ECOMMERCE,"Martinez, Nielsen and Miller",,contact@martineznielsenandmi3.com,+1 834 570 9459,959 Janet Cape Apt. 413
2,GT-000003,ERP,"Martinez, Nielsen and Miller",34-1166941,contact@martineznielsenandmi3.com,834-570-9459,959 Janet Cape Apt. 413
908,GT-000005,ECOMMERCE,Trujillo Group,,contact@trujillogroup5.com,+1 611 474 1084,39537 Wilkerson Row Suite 242
4,GT-000005,ERP,Trujillo Group,37-9515702,contact@trujillogroup5.com,611-474-1084,39537 Wilkerson Row Suite 242
